# Zadanie 4: graf przepisów i agent z Cypherem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adrianstando/2026Z-PJATK-TEG/blob/main/04-graphrag/zadanie.ipynb)

**Oddanie:** mail z tematem `[TEG] lab-04`, w załączniku ten notebook **z wynikami komórek** i folder `przepisy/` (pliki z tekstami przepisów), bez kluczy API i haseł.

**Termin:** przed zajęciami 5.

Graf wiedzy z pięciu przepisów kulinarnych: LLM zamienia tekst przepisu na węzły i relacje według podanego schematu, a agent ReAct z jednym narzędziem do zapytań Cypher podpowiada, co można ugotować z posiadanych składników.

Części:
1. Własna baza Neo4j i połączenie z notebooka.
2. Pięć przepisów jako pliki tekstowe.
3. Ekstrakcja grafu przez LLM i załadowanie do Neo4j.
4. Agent z narzędziem `run_cypher` i dwa pytania.

Kod z [notebooka z zajęć](notebook.ipynb) można kopiować i modyfikować.

## 0. Konfiguracja

In [ ]:
# Colab / Deepnote / Codespaces: odkomentuj i uruchom raz
# %pip install -q openai neo4j pydantic python-dotenv langchain langchain-openai langgraph

Dostawcę modelu wybiera się zmienną `LLM_PROVIDER`, tak jak w notebooku z zajęć ([00-dostep-do-llm](../00-dostep-do-llm/README.md)). Zadanie nie wymaga embeddingów. Ekstrakcja korzysta ze structured outputs, a agent z wywołań narzędzi; małe modele lokalne częściej gubią pola i mylą argumenty narzędzi.

In [ ]:
import os

try:  # Colab: sekrety z panelu "Secrets"
    from google.colab import userdata
    for k in ["LLM_PROVIDER", "OLLAMA_CHAT", "OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY",
              "DEEPSEEK_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]:
        try:
            os.environ.setdefault(k, userdata.get(k))
        except Exception:
            pass
except ImportError:  # lokalnie: plik .env
    from dotenv import load_dotenv
    load_dotenv(override=True)

from openai import OpenAI, AzureOpenAI
from langchain_openai import ChatOpenAI, AzureChatOpenAI

MODELS = {
    "openai":   {"chat": "gpt-5-nano", "agent": "gpt-5.4-nano"},
    "azure":    {"chat": "gpt-4o-mini", "agent": "gpt-4o-mini"},
    "deepseek": {"chat": "deepseek-chat", "agent": "deepseek-chat"},
    "ollama":   {"chat": os.getenv("OLLAMA_CHAT", "gemma4:e4b"), "agent": "gemma4:e4b"},
    "proxy":    {"chat": "copilot/gpt-5-mini", "agent": "copilot/gpt-5-mini"},
}

def make_client(provider):
    if provider == "azure":
        return AzureOpenAI(api_version="2024-12-01-preview")
    if provider == "deepseek":
        return OpenAI(base_url="https://api.deepseek.com", api_key=os.environ["DEEPSEEK_API_KEY"])
    if provider == "ollama":
        return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    if provider == "proxy":
        return OpenAI(base_url="http://localhost:8787/v1", api_key="teg")
    return OpenAI()

PROVIDER = os.getenv("LLM_PROVIDER", "ollama")
client = make_client(PROVIDER)
CHAT, AGENT = MODELS[PROVIDER]["chat"], MODELS[PROVIDER]["agent"]  # CHAT: ekstrakcja, AGENT: pętla agenta

def make_llm(model):
    if PROVIDER == "azure":
        return AzureChatOpenAI(azure_deployment=model, api_version="2024-12-01-preview")
    return ChatOpenAI(model=model, base_url=str(client.base_url), api_key=client.api_key)

print(f"czat: {PROVIDER} ({CHAT}, agent: {AGENT})")

## Część 1. Neo4j

Dwa sposoby na własną bazę (do wyboru):

1. **Docker Compose**, lokalnie albo w chmurze (np. GitHub Codespaces): `docker compose up -d` w folderze `04-graphrag` (plik [docker-compose.yml](docker-compose.yml)). Adres `bolt://localhost:7687`, logowanie `neo4j` / `password123`, podgląd grafu: http://localhost:7474.
2. **Neo4j AuraDB Free**: https://console.neo4j.io → *Create instance* → *Free*. Przy tworzeniu instancji pobiera się plik `.txt` z danymi logowania: `NEO4J_URI` (`neo4j+s://…`), `NEO4J_USERNAME`, `NEO4J_PASSWORD`. Te wartości trzeba wpisać do `.env` albo do *Secrets* w Colabie, nie do kodu.

Zadanie:
1. Połączenie przez oficjalny sterownik `neo4j` (`GraphDatabase.driver`, `verify_connectivity()`), adres i hasło ze zmiennych środowiskowych.
2. Funkcja `cypher(query, **params)` zwracająca listę słowników oraz funkcja `run_read(query)`, która wykonuje zapytanie w transakcji **tylko do odczytu** (`session.execute_read`), jak w notebooku z zajęć.
3. Wypisanie wersji serwera: `CALL dbms.components() YIELD name, versions RETURN name, versions[0] AS version`.

In [ ]:
from neo4j import GraphDatabase

# TODO: driver, verify_connectivity(), cypher(), run_read(), wersja serwera

## Część 2. Przepisy

1. Pięć przepisów ze strony [aniagotuje.pl](https://aniagotuje.pl), np. świątecznych (pierniki, makowiec, sernik, barszcz, pierogi). Przynajmniej dwa przepisy powinny mieć część wspólnych składników (mąka, jajka, cukier, masło…), żeby graf miał połączenia.
2. Każdy przepis to osobny plik w folderze `przepisy/` (`.md` albo `.txt`), z tekstem skopiowanym ze strony: tytuł, składniki z ilościami, kroki przygotowania, czas, liczba porcji. **Pierwsza linia pliku: `URL: <adres oryginalnego przepisu>`.** Tekstu nie trzeba przepisywać ani poprawiać; reklamy i komentarze można pominąć.
3. Wczytanie plików: słownik `{nazwa pliku: tekst}`, wypisana liczba plików i adres URL każdego przepisu.

In [ ]:
from pathlib import Path

# TODO: wczytanie przepisy/*.md|txt, liczba plików, URL każdego przepisu

## Część 3. Ekstrakcja i graf

Docelowy schemat grafu:

```
(:Recipe {title, url, prep_time_min, servings})
(:Recipe)-[:CONTAINS {amount, unit}]->(:Ingredient {name})
(:Recipe)-[:IN_CATEGORY]->(:Category {name})
(:Recipe)-[:COOKED_BY]->(:Method {name})
(:Recipe)-[:HAS_STEP]->(:Step {number, text})
```

Klasy Pydantic poniżej opisują tylko część tego schematu: składniki, kroki i tytuł przepisu.

Zadanie:
1. Uzupełnienie klasy `Recipe` o pola potrzebne do pozostałych węzłów i właściwości z docelowego schematu: kategoria dania, sposób przygotowania, czas przygotowania w minutach, liczba porcji. Każde pole z opisem `Field(description=...)`, bo opisy trafiają do promptu.
2. Prompt systemowy i funkcja `extract(text)`, która przez structured outputs (`client.chat.completions.parse(..., response_format=Recipe)`) zamienia tekst przepisu na obiekt `Recipe`. Adres URL pochodzi z pierwszej linii pliku, nie od modelu.
3. Ekstrakcja wszystkich pięciu przepisów; wypisany wynik dla jednego (`model_dump_json(indent=1)`).
4. Załadowanie do Neo4j według schematu grafu: ograniczenia `UNIQUE` na kluczach (`Recipe.url`, `Ingredient.name`, `Category.name`, `Method.name`), `MERGE` zamiast `CREATE`, parametry (`$...`) zamiast sklejania tekstu zapytania. Węzły `Step` należą do jednego przepisu: kluczem może być para (url przepisu, numer kroku).
5. Wypisanie liczby węzłów każdej etykiety, liczby relacji i **posortowanej listy wszystkich nazw składników** (`MATCH (i:Ingredient) RETURN i.name ORDER BY i.name`).

In [ ]:
from pydantic import BaseModel, Field

class Ingredient(BaseModel):
    name: str = Field(description="nazwa składnika")
    amount: float | None = Field(description="ilość, jeśli podana")
    unit: str | None = Field(description="jednostka, np. g, ml, szt., łyżka; None, jeśli brak")

class Step(BaseModel):
    number: int = Field(description="numer kroku, od 1")
    text: str = Field(description="opis kroku przygotowania")

class Recipe(BaseModel):
    title: str = Field(description="tytuł przepisu")
    ingredients: list[Ingredient]
    steps: list[Step]
    # TODO: pola dla Category, Method, prep_time_min i servings

In [ ]:
# TODO: prompt systemowy, extract(text), ekstrakcja pięciu przepisów, wynik dla jednego

In [ ]:
# TODO: ograniczenia UNIQUE, ładowanie przez MERGE, liczby węzłów i relacji, lista nazw składników

## Część 4. Agent

Agent ReAct z **jednym narzędziem** `run_cypher(query: str)`. Przy jednym narzędziu wystarczy gotowa pętla `create_agent` z LangChain (sekcja 6 notebooka z zajęć 3); własny graf `StateGraph` w LangGraph (sekcja 8 notebooka z zajęć 4) też jest w porządku.

1. Narzędzie wykonuje zapytanie przez `run_read` (tylko odczyt) i zwraca wiersze jako JSON; błąd zwraca jako tekst, żeby model mógł poprawić zapytanie.
2. System prompt zawiera schemat grafu z części 3 i opisuje, jak agent ma pracować (np. najpierw sprawdzić, jakie nazwy składników są w grafie, a dopiero potem szukać przepisów).
3. Dwa pytania, przy każdym wypisane kroki: zapytania Cypher od modelu, skrócone wyniki i końcowa odpowiedź.
   - „Mam mąkę, jajka i cukier. Co mogę z tego upiec?”
   - drugie pytanie własne, np. o czas przygotowania, kategorię albo przepis bez wybranego składnika.
4. W komórce markdown (2–3 zdania): czy odpowiedzi agenta zgadzają się z przepisami i co ewentualnie poszło źle.

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent  # albo StateGraph, ToolNode, tools_condition z langgraph

# TODO: run_cypher, system prompt ze schematem grafu, agent

In [ ]:
# TODO: pytanie 1, wypisane kroki i odpowiedź

In [ ]:
# TODO: pytanie 2, wypisane kroki i odpowiedź

*Ocena odpowiedzi agenta:*